# Análise Qualitativa por Classe de Uso do Solo

Cada célula abaixo é responsável por **uma classe**. Em cada uma, são exibidas **todas** as imagens onde a classe aparece, organizadas em um grid de **5 colunas** (fileiras de 5 imagens que "pulam pra baixo").

- Borda vermelha delimita a área da classe
- Sobreposição branca semi-transparente destaca a região
- Título de cada imagem mostra o nome do arquivo e a % de ocupação
- Imagens ordenadas por ocupação (maior primeiro)
- **Dica**: se ficar pesado, defina `MAX_IMAGES` na célula de configuração (ex: `MAX_IMAGES = 30`)


In [ ]:
import os
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from tqdm import tqdm

# === CONFIGURACAO ===
# Ajuste os caminhos para as suas pastas de mascaras coloridas e imagens de satelite
MASKS_DIR  = r"D:\Imagens CAR - Testes 11-02 COM COR"
RAW_DIR    = r"D:\Imagens CAR - Testes 11-02 SEM COR"
N_COLS     = 5    # Colunas por fileira no grid
MAX_IMAGES = None # None = exibe TODAS as imagens; defina um numero para limitar (ex: 50)

# Mapeamento completo de classes (25 classes)
CLASSES = {
    "Afloramento Rochoso":                           (150, 150, 150),
    "Área Edificada":                                 (251, 154, 153),
    "Brejo":                                          ( 69, 175, 213),
    "Campo Rupestre/Altitude":                        (150, 109, 207),
    "Cultivo Agrícola - Abacaxi":                     (128, 214,  16),
    "Cultivo Agrícola - Banana":                      (247, 223,   8),
    "Cultivo Agrícola - Café":                        (119,   9,  29),
    "Cultivo Agrícola - Cana-De-Açúcar":              (209, 163, 117),
    "Cultivo Agrícola - Coco-Da-Baía":                (231,  67,  97),
    "Cultivo Agrícola - Mamão":                       (245, 141,  23),
    "Cultivo Agrícola - Outros Cultivos Permanentes": ( 55, 196, 201),
    "Cultivo Agrícola - Outros Cultivos Temporários": (225, 175,  38),
    "Extração Mineração":                             ( 81,  77,  77),
    "Macega":                                         (211, 127, 122),
    "Mangue":                                         (156,  68, 203),
    "Massa D'Água":                                   (133, 196, 221),
    "Mata Nativa":                                    ( 13, 103,  19),
    "Mata Nativa em Estágio Inicial de Regeneração":  ( 51, 160,  44),
    "Outros":                                         ( 31, 205, 170),
    "Pastagem":                                       (178, 214,  32),
    "Reflorestamento - Eucalipto":                    (207, 103,  65),
    "Reflorestamento - Pinus":                        (243, 184, 129),
    "Reflorestamento - Seringueira":                  (151, 132, 233),
    "Restinga":                                       ( 63, 231, 161),
    "Solo Exposto":                                   (245, 222, 193),
}

print(f"Classes disponiveis: {len(CLASSES)}")
print(f"Colunas por fileira: {N_COLS}")
print(f"Limite de imagens:   {'Sem limite (todas)' if MAX_IMAGES is None else MAX_IMAGES}")


In [ ]:
# === FUNCOES AUXILIARES ===

def find_class_in_mask(tif_path, target_rgb):
    """Abre a mascara TIF e retorna onde a classe aparece."""
    with rasterio.open(tif_path) as src:
        img = src.read()
    img_rgb = np.transpose(img[:3], (1, 2, 0))
    mascara = np.all(img_rgb == np.array(target_rgb), axis=-1)
    return mascara, img_rgb


def load_raw_image(raw_path):
    """Abre a imagem de satelite pura."""
    with rasterio.open(raw_path) as src:
        img = src.read()
    return np.transpose(img[:3], (1, 2, 0))


def highlight_on_raw(raw_rgb, mascara, cor_borda=(255, 0, 0), espessura=4, alpha=0.15):
    """Desenha borda vermelha e sobreposicao leve na area da classe."""
    resultado = raw_rgb.copy().astype(np.float32)
    resultado[mascara] = resultado[mascara] * (1 - alpha) + np.array([255, 255, 255], dtype=np.float32) * alpha
    resultado = np.clip(resultado, 0, 255).astype(np.uint8)
    contorno = binary_dilation(mascara, iterations=espessura) & ~mascara
    resultado[contorno] = list(cor_borda)
    return resultado


def calc_occupation(mascara):
    """Retorna a % de pixels da classe."""
    return 100.0 * mascara.sum() / mascara.size


print("Funcoes carregadas.")


In [ ]:
# === INDEXACAO: escaneia todas as imagens UMA VEZ e mapeia quais classes aparecem em cada ===

mask_files = sorted(glob.glob(os.path.join(MASKS_DIR, "*.tif")))

# Montar lista de pares (mascara, satelite)
matched_files = []
for mf in mask_files:
    nome = os.path.basename(mf)
    raw_path = os.path.join(RAW_DIR, nome)
    if os.path.exists(raw_path):
        matched_files.append((mf, raw_path, nome))

print(f"Pares mascara+satelite encontrados: {len(matched_files)}")

# Indexar: para cada classe, guardar lista de (mask_path, raw_path, nome, ocupacao)
class_index = {name: [] for name in CLASSES}

for mask_path, raw_path, nome in tqdm(matched_files, desc="Indexando classes"):
    with rasterio.open(mask_path) as src:
        img = src.read()
    img_rgb = np.transpose(img[:3], (1, 2, 0))
    total_pixels = img_rgb.shape[0] * img_rgb.shape[1]

    for class_name, class_rgb in CLASSES.items():
        mascara = np.all(img_rgb == np.array(class_rgb), axis=-1)
        n_pixels = mascara.sum()
        if n_pixels > 0:
            occupation = 100.0 * n_pixels / total_pixels
            class_index[class_name].append({
                "mask_path": mask_path,
                "raw_path": raw_path,
                "nome": nome,
                "occupation": occupation
            })

# Ordenar cada classe por ocupacao (maior primeiro) para pegar os melhores exemplos
for class_name in class_index:
    class_index[class_name].sort(key=lambda x: x["occupation"], reverse=True)

print("\nResumo da indexacao:")
print(f"{'Classe':<55} {'Imagens':>8} {'Ocup. media':>12}")
print("-" * 78)
for class_name, entries in class_index.items():
    n = len(entries)
    media = np.mean([e["occupation"] for e in entries]) if n > 0 else 0
    print(f"{class_name:<55} {n:>8} {media:>11.2f}%")


In [ ]:
def show_class_grid(class_name, n_cols=N_COLS, max_images=MAX_IMAGES):
    """
    Exibe TODAS as imagens de uma classe em um grid de N_COLS colunas.
    As imagens sao ordenadas por ocupacao (maior primeiro).
    Se max_images for definido, limita a quantidade exibida.
    """
    entries = class_index[class_name]
    rgb = CLASSES[class_name]

    if len(entries) == 0:
        print(f"  Classe '{class_name}' nao encontrada em nenhuma imagem.")
        return

    # Aplicar limite se definido
    samples = entries if max_images is None else entries[:max_images]
    n = len(samples)
    n_rows = (n + n_cols - 1) // n_cols  # ceil division

    print(f"  Exibindo {n} imagens ({n_rows} fileiras x {n_cols} colunas)...")

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4.5 * n_rows))

    # Garantir que axes seja sempre 2D
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_cols == 1:
        axes = axes[:, np.newaxis]

    for i, entry in enumerate(samples):
        row = i // n_cols
        col = i % n_cols

        mascara, _ = find_class_in_mask(entry["mask_path"], rgb)
        raw_rgb = load_raw_image(entry["raw_path"])
        highlighted = highlight_on_raw(raw_rgb, mascara)

        axes[row, col].imshow(highlighted)
        nome_curto = entry["nome"].replace(".tif", "")
        if len(nome_curto) > 25:
            nome_curto = nome_curto[:12] + "..." + nome_curto[-10:]
        axes[row, col].set_title(f"{nome_curto}\n{entry['occupation']:.1f}%", fontsize=8)
        axes[row, col].axis("off")

    # Desligar eixos vazios na ultima fileira
    for j in range(n, n_rows * n_cols):
        row = j // n_cols
        col = j % n_cols
        axes[row, col].axis("off")

    aparicao = 100.0 * len(entries) / len(matched_files)
    limitado = f" (mostrando {n}/{len(entries)})" if max_images and n < len(entries) else ""
    fig.suptitle(
        f"{class_name}  |  RGB {rgb}  |  {len(entries)}/{len(matched_files)} imagens ({aparicao:.1f}%){limitado}",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    plt.show()
    plt.close(fig)  # Liberar memoria


print("Funcao show_class_grid pronta.")


---
## Visualização por Classe

Cada célula abaixo gera o grid de uma classe específica.


In [ ]:
show_class_grid("Afloramento Rochoso")


In [ ]:
show_class_grid("Área Edificada")


In [ ]:
show_class_grid("Brejo")


In [ ]:
show_class_grid("Campo Rupestre/Altitude")


In [ ]:
show_class_grid("Cultivo Agrícola - Abacaxi")


In [ ]:
show_class_grid("Cultivo Agrícola - Banana")


In [ ]:
show_class_grid("Cultivo Agrícola - Café")


In [ ]:
show_class_grid("Cultivo Agrícola - Cana-De-Açúcar")


In [ ]:
show_class_grid("Cultivo Agrícola - Coco-Da-Baía")


In [ ]:
show_class_grid("Cultivo Agrícola - Mamão")


In [ ]:
show_class_grid("Cultivo Agrícola - Outros Cultivos Permanentes")


In [ ]:
show_class_grid("Cultivo Agrícola - Outros Cultivos Temporários")


In [ ]:
show_class_grid("Extração Mineração")


In [ ]:
show_class_grid("Macega")


In [ ]:
show_class_grid("Mangue")


In [ ]:
show_class_grid("Massa D'Água")


In [ ]:
show_class_grid("Mata Nativa")


In [ ]:
show_class_grid("Mata Nativa em Estágio Inicial de Regeneração")


In [ ]:
show_class_grid("Outros")


In [ ]:
show_class_grid("Pastagem")


In [ ]:
show_class_grid("Reflorestamento - Eucalipto")


In [ ]:
show_class_grid("Reflorestamento - Pinus")


In [ ]:
show_class_grid("Reflorestamento - Seringueira")


In [ ]:
show_class_grid("Restinga")


In [ ]:
show_class_grid("Solo Exposto")
